# Modelos Chat (LLMs)

### Empezamos cargando los modelos

usaremos estos modelos (pequeños 1B-3B y medianos 7B-9B):
    "Qwen2.5-1.5B-Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen2.5-7B-Instruct": "Qwen/Qwen2.5-7B-Instruct",


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

# Carga las variables de entorno desde .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN") or None

# Carpeta donde se guardarán los modelos
BASE_DIR = Path("/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Chat")
BASE_DIR.mkdir(parents=True, exist_ok=True)

models_to_download = {
    "Qwen2.5-1.5B-Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen2.5-7B-Instruct": "Qwen/Qwen2.5-7B-Instruct",
}

# Patrones de archivos innecesarios a ignorar
ignore = ["*.onnx*", "onnx/*", "imgs/*", "*.jpg", "*.jpeg", "*.webp", "*.png", "*.msgpack*", "*.h5*", "*.ot"]

# Descarga optimizada: solo los pesos esenciales, tokenizadores y configs
for name, model_id in models_to_download.items():
    target_dir = BASE_DIR / name
    print(f"Descargando {name} ({model_id}) en {target_dir}...")
    snapshot_download(
        repo_id=model_id,
        local_dir=str(target_dir),
        ignore_patterns=ignore,
        token=hf_token
    )
    print(f"{name} descargado en {target_dir}")


# Carga de modelos en ram / vram


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Elige el modelo que quieres cargar
model_name = "Qwen2.5-1.5B-Instruct"  # opciones: "Qwen2.5-1.5B-Instruct", "Qwen2.5-7B-Instruct"

# Carga el modelo y tokenizador desde la carpeta local
model_path = f"/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Chat/{model_name}"
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

print(f"✓ {model_name} cargado correctamente en memoria")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 2. Elige los modelos que quieres cargar para el benchmark
selected_models = ["Qwen2.5-1.5B-Instruct"]

models = {}
tokenizers = {}
for name in selected_models:
    path = f"/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Chat/{name}"
    tokenizers[name] = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    models[name] = AutoModelForCausalLM.from_pretrained(
        path,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True
    )
    print(f"✓ {name} listo en memoria")
